In [ ]:
print("Startup Job Finder – discovery layer ready")

Setup

In [ ]:
from tavily import TavilyClient
from dotenv import load_dotenv
import os

load_dotenv()
client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Company discovery query

In [ ]:
query = (
  '"AI startup" '
  '("about us" OR "company" OR "who we are") '
  '-news -blog -article -wikipedia -linkedin -medium'
)

response = client.search(
    query=query,
    search_depth="advanced",
    max_results=20
)

for r in response["results"]:
    print("TITLE:", r["title"])
    print("URL:", r["url"])
    print("----")


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

CAREER_KEYWORDS = [
    "career",
    "job",
    "join",
    "work with",
    "hiring",
]

BAD_HINTS = ["contact", "about", "privacy", "terms", "press", "blog"]

def looks_like_careers(text: str, href: str) -> bool:
    t = text.lower()
    h = href.lower()

    # reject anchors like "#Contact"
    if h.startswith("#"):
        return False

    # reject obvious non-careers pages
    if any(b in t or b in h for b in BAD_HINTS):
        return False

    # accept careers-ish
    return any(k in t or k in h for k in CAREER_KEYWORDS)


def find_careers_url(homepage: str):
    try:
        resp = requests.get(homepage, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        for a in soup.find_all("a", href=True):
            text = (a.get_text() or "").lower()
            href = a["href"].lower()

            if looks_like_careers(text, href):
                url = urljoin(homepage, href)
                return url.rstrip("/")

    except Exception as e:
        print("ERROR:", homepage, e)

    return None


In [ ]:
companies = [
    "https://alphaai.biz",
    "https://ai-nation.de",
    "https://apera.ai",
]

for c in companies:
    print(c, "→", find_careers_url(c))
